In [22]:

import numpy as np
from  PIL import Image

import torch
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision import datasets, transforms

run_name = "run3_5stage" # give every experiment a new name

In [23]:
train_path = "imagewoof_train/train"
val_path = "imagewoof_val/val"

In [24]:
MEAN = (0.5, 0.5, 0.5)
STD  = (0.5, 0.5, 0.5)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(128, scale=(0.25, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.2)),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

In [25]:
train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
val_dataset   = datasets.ImageFolder(root=val_path,   transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle= True, num_workers= 4, pin_memory= True, persistent_workers= True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle= True, num_workers= 2)

#checking the data sape and the size
#image, label = train_dataset[0]
#image 
# image.size() #torch.Size([3, 128, 128])

In [26]:
#Loads the orderd class name from classes.txt
with open("classes.txt") as f:
    class_name = [line.strip() for line in f if line.strip()]

print(class_name)

['0 shih_tzu', '1 rhodesian_ridgeback', '2 beagle', '3 english_foxhound', '4 border_terrier', '5 australian_terrier', '6 golden_retriever', '7 old_english_sheepdog', '8 samoyed', '9 dingo']


In [27]:
import torch.nn as nn
"""
*block(...) unpacks the three layers (Conv → BN → ReLU) into the list, so the whole thing is still one plain nn.Sequential. 
There are 13 conv layers, about 9M parameters, and no skip connections, so it’s within the rules.
"""
class NeuralNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        def block(cin, cout):
            return [nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                    nn.BatchNorm2d(cout),
                    nn.ReLU(inplace=True)]

        self.layers = nn.Sequential(
            *block(3, 64),    *block(64, 64),                       nn.MaxPool2d(2),  # 128 -> 64
            *block(64, 128),  *block(128, 128),                     nn.MaxPool2d(2),  # 64 -> 32
            *block(128, 256), *block(256, 256), *block(256, 256),   nn.MaxPool2d(2),  # 32 -> 16
            *block(256, 384), *block(384, 384), *block(384, 384),   nn.MaxPool2d(2),  # 16 -> 8
            *block(384, 512), *block(512, 512), *block(512, 512),   nn.MaxPool2d(2),  # 8 -> 4
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.layers(x)

model = NeuralNet()
x = image.unsqueeze(0)      # [1, 3, 128, 128]
out = model(x)
print(out.shape)            # torch.Size([1, 10])

In [28]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

import sys, torch, subprocess
print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("Torch built for CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout[:800])

In [29]:
import time
import torch
import torch.nn as nn

# ---- settings for this run ----
num_epochs = 150

# ---- device ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Training on:", device)

# ---- fresh model, loss, optimizer, scheduler ----
torch.manual_seed(42)
net = NeuralNet().to(device)
print(f"Parameters: {sum(p.numel() for p in net.parameters())/1e6:.2f}M")   # should be ~9M



loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

scaler = torch.amp.GradScaler("cuda")

best_acc = 0.0
start_time = time.time()


for epoch in range(num_epochs):
    epoch_start = time.time()

    # ---- training ----
    net.train()
    running_loss = 0.0
    correct, total = 0, 0

    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = net(inputs)
            loss = loss_function(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # ---- validation ----
    net.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = net(inputs)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total

    # ---- learning-rate schedule ----
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # ---- timer ----
    epoch_time = time.time() - epoch_start
    elapsed = time.time() - start_time
    remaining = elapsed / (epoch + 1) * (num_epochs - epoch - 1)

    print(f"Epoch {epoch+1}/{num_epochs} | loss {train_loss:.3f} | train acc {train_acc:.3f} | "
          f"val acc {val_acc:.3f} | lr {current_lr:.4f} | "
          f"{epoch_time:.0f}s | elapsed {elapsed/60:.1f} min | ~{remaining/60:.0f} min left")

    # ---- save the best model ----
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(net.state_dict(), f"{run_name}.pth")
        print(f"  saved new best model to {run_name}.pth")

total_time = time.time() - start_time
print(f"Done in {total_time/60:.1f} min. Best val accuracy: {best_acc:.3f}")



Training on: cuda
Parameters: 11.77M
Epoch 1/150 | loss 3.061 | train acc 0.114 | val acc 0.123 | lr 0.1000 | 27s | elapsed 0.4 min | ~66 min left
  saved new best model to run3_5stage.pth
Epoch 2/150 | loss 2.266 | train acc 0.153 | val acc 0.149 | lr 0.1000 | 10s | elapsed 0.6 min | ~45 min left
  saved new best model to run3_5stage.pth
Epoch 3/150 | loss 2.196 | train acc 0.172 | val acc 0.198 | lr 0.1000 | 10s | elapsed 0.8 min | ~38 min left
  saved new best model to run3_5stage.pth
Epoch 4/150 | loss 2.185 | train acc 0.189 | val acc 0.212 | lr 0.0999 | 10s | elapsed 0.9 min | ~34 min left
  saved new best model to run3_5stage.pth
Epoch 5/150 | loss 2.176 | train acc 0.199 | val acc 0.198 | lr 0.0998 | 10s | elapsed 1.1 min | ~32 min left
Epoch 6/150 | loss 2.164 | train acc 0.202 | val acc 0.232 | lr 0.0997 | 11s | elapsed 1.3 min | ~31 min left
  saved new best model to run3_5stage.pth
Epoch 7/150 | loss 2.148 | train acc 0.210 | val acc 0.221 | lr 0.0996 | 10s | elapsed 1.4 mi

# To copy the best one to model.pth
import shutil
shutil.copy("run2_5stage.pth", "model.pth")   # use your best run's name